# Lab04: Feature Selection (Nhóm)

## Dataset House Price Prediction

## Tiền xử lý dữ liệu

In [ ]:
import pandas as pd

train_path = 'data/train.csv'
test_path = 'data/test.csv'

# Đọc dữ liệu vào DataFrame
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

display(df_train.head())

In [ ]:
print("Kích thước train:", df_train.shape)
print("Kích thước test:", df_test.shape)

# Số lượng giá trị thiếu trong các cột
missing = df_train.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].head(20))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

target = "SalePrice"

# Tách đặc trưng và giá cần dự đoán; không dùng Id làm đặc trưng
X = df_train.drop(columns=[target, "Id"], errors="ignore")
y = df_train[target]

# Chia để kiểm tra mô hình trên dữ liệu chưa dùng để huấn luyện
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_columns = X_train.select_dtypes(include="number").columns
categorical_columns = X_train.select_dtypes(exclude="number").columns

numeric_preprocessing = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median"))
])

categorical_preprocessing = Pipeline([
    ("fill_missing", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessing, numeric_columns),
    ("categorical", categorical_preprocessing, categorical_columns)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

valid_predictions = model.predict(X_valid)
rmse = mean_squared_error(y_valid, valid_predictions) ** 0.5
print("Validation RMSE:", rmse)

In [ ]:
X_test = df_test.drop(columns=["Id"], errors="ignore")
X_test = X_test.reindex(columns=X.columns)

model.fit(X, y)
test_predictions = model.predict(X_test)

submission = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": test_predictions
})

submission.to_csv("data/submission.csv", index=False)

## SVM

## XGBoost


## RandomForest


In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

### RandomForest cơ bản

In [ ]:
model_rf = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_valid)

rmse_rf = mean_squared_error(
    y_valid,
    pred_rf
) ** 0.5

print("RandomForest RMSE:", rmse_rf)

### RandomForest + Feature Selection

In [ ]:
model_rf_fs = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(
        score_func=f_regression,
        k=100
    )),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model_rf_fs.fit(X_train, y_train)

pred_rf_fs = model_rf_fs.predict(X_valid)

rmse_rf_fs = mean_squared_error(
    y_valid,
    pred_rf_fs
) ** 0.5

print("RandomForest + Feature Selection RMSE:", rmse_rf_fs)